# BRONZE → SILVER

In [9]:
# ============================================================
# BRONZE → SILVER
# Enterprise Finance Medallion Architecture
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================================
# READ BRONZE TABLE
# ============================================================

bronze_table_path = "abfss://MariaMonedero@onelake.dfs.fabric.microsoft.com/lh_enterprise_bronze.Lakehouse/Tables/dbo/bronze_fact_operations_raw"

df = spark.read.format("delta").load(bronze_table_path)

display(df)

StatementMeta(, 91c94281-14df-4329-a547-61642fa7fc44, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 60960264-feea-4bd5-ac79-6d8647f66d96)

In [10]:
# ============================================================
# DATA CLEANING & VALIDATION
# ============================================================

# Normalize column names
for old_name in df.columns:
    new_name = (
        old_name.strip()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )
    df = df.withColumnRenamed(old_name, new_name)

# Text columns
text_columns = [
    "Region",
    "Department",
    "Business_Unit",
    "Project_Status",
    "Risk_Level",
    "Project_Manager"
]

# Clean text values
for c in text_columns:
    if c in df.columns:
        df = df.withColumn(c, initcap(trim(col(c))))

# Standardize Department values
if "Department" in df.columns:
    df = df.withColumn(
        "Department",
        when(upper(trim(col("Department"))).isin("FIN", "FIN.", "FINANCE"), "Finance")
        .when(upper(trim(col("Department"))).isin("CS", "CUSTOMER SERVICE", "CUSTOMER_SERVICE", "COSTUMER SERVICE", "COSTUMER_SERVICE"), "Customer Service")
        .when(upper(trim(col("Department"))).isin("HR", "HUMAN RESOURCES"), "Human Resources")
        .when(upper(trim(col("Department"))).isin("OPS", "OPERATIONS"), "Operations")
        .when(upper(trim(col("Department"))).isin("IT", "I.T.", "I T"), "IT")
        .otherwise(initcap(trim(col("Department"))))
    )


# Standardize Region values
if "Region" in df.columns:
    df = df.withColumn(
        "Region",
        when(
            upper(trim(col("Region"))).isin(
                "UK IRELAND",
                "UK I",
                "UK&I",
                "UK & IRELAND",
                "UK AND IRELAND"
            ),
            "UK & Ireland"
        )
        .otherwise(trim(col("Region")))
    )

# Standardize Business Unit values
if "Business_Unit" in df.columns:
    df = df.withColumn(
        "Business_Unit",
        when(col("Business_Unit").isNull() | (trim(col("Business_Unit")) == ""), "Other")
        .otherwise(initcap(trim(col("Business_Unit"))))
    )


# ============================================================
# TYPE CASTING
# ============================================================

decimal_columns = [
    "Revenue",
    "Operating_Cost",
    "Budget",
    "Forecast",
    "Productivity_Score",
    "Satisfaction_Index",
    "Hidden_Operational_Cost"
]

for c in decimal_columns:
    if c in df.columns:
        df = df.withColumn(
            c,
            regexp_replace(col(c), ",", ".").cast(DoubleType())
        )

if "Employee_Count" in df.columns:
    df = df.withColumn(
        "Employee_Count",
        col("Employee_Count").cast(IntegerType())
    )

if "Date" in df.columns:
    df = df.withColumn(
        "Date",
        to_date(col("Date"), "yyyy-MM-dd")
    )


# ============================================================
# DATA QUALITY
# ============================================================

rows_before = df.count()

df = df.dropDuplicates()

df = df.dropna(subset=[
    "Revenue",
    "Operating_Cost",
    "Budget",
    "Forecast"
])

rows_after = df.count()
rows_removed = rows_before - rows_after


# ============================================================
# KPI ENGINEERING
# ============================================================

df = df.withColumn(
    "Profit",
    col("Revenue") - col("Operating_Cost")
)

df = df.withColumn(
    "Budget_Variance",
    col("Revenue") - col("Budget")
)

df = df.withColumn(
    "Forecast_Variance",
    col("Revenue") - col("Forecast")
)

df = df.withColumn(
    "Profit_Margin",
    when(
        col("Revenue") != 0,
        round((col("Profit") / col("Revenue")) * 100, 2)
    ).otherwise(None)
)

# Recalculate Risk Level based on Profit Margin
df = df.withColumn(
    "Risk_Level",
    when(col("Profit_Margin").isNull(), "Unknown")
    .when(col("Profit_Margin") < 15, "Critical")
    .when(col("Profit_Margin") < 25, "High")
    .when(col("Profit_Margin") < 40, "Medium")
    .otherwise("Low")
)

df = df.withColumn("silver_processed_timestamp", current_timestamp())

display(df)

StatementMeta(, 91c94281-14df-4329-a547-61642fa7fc44, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 529ffac1-ddb5-44e1-86a6-fd06aa01db06)

In [11]:
# ============================================================
# SAVE SILVER TABLE
# ============================================================

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_fact_operations")

print("========================================")
print("SILVER TABLE CREATED SUCCESSFULLY")
print("========================================")
print("Table name: silver_fact_operations")
print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning: {rows_after}")
print(f"Rows removed: {rows_removed}")
print(f"Columns available: {len(df.columns)}")

StatementMeta(, 91c94281-14df-4329-a547-61642fa7fc44, 13, Finished, Available, Finished, False)

SILVER TABLE CREATED SUCCESSFULLY
Table name: silver_fact_operations
Rows before cleaning: 30000
Rows after cleaning: 28777
Rows removed: 1223
Columns available: 26
